<a href="https://colab.research.google.com/github/SrEffry/Mamografias/blob/main/02_eda_particion.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 02 · EDA tabular, índice DICOM y partición por paciente

> **Prototipo académico; no apto para diagnóstico ni decisiones clínicas.**

Versión depurada: se quitaron las celdas obsoletas (cruce por carpeta y por manifiestos, lectura con `force=True`, `%%writefile`). Ejecutar de arriba abajo.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
BASE = "/content/drive/MyDrive/Aprendizaje final"
DATA = f"{BASE}/Csv"
ROOT = f"{BASE}/Images"
SRC  = f"{BASE}/src"
RES  = f"{BASE}/resultados"
os.makedirs(SRC, exist_ok=True); os.makedirs(RES, exist_ok=True)
print("CSV en DATA:", sorted(os.listdir(DATA)))

## Carga y limpieza de los CSV de masas

In [ ]:
import pandas as pd, numpy as np
import matplotlib.pyplot as plt

# Ajustado automáticamente para incluir la subcarpeta correcta encontrada en tu Drive
DATA = "/content/drive/MyDrive/Aprendizaje final/Csv"

def cargar(archivo, particion):
    df = pd.read_csv(f"{DATA}/{archivo}")
    df = df.rename(columns={"breast density": "breast_density"})
    for c in ["image file path", "cropped image file path", "ROI mask file path"]:
        df[c] = df[c].str.strip()          # las rutas traen saltos de línea al final
    df["label"] = (df["pathology"] == "MALIGNANT").astype(int)
    df["lesion_id"] = (df["patient_id"] + "_" + df["left or right breast"] + "_"
                       + df["image view"] + "_" + df["abnormality id"].astype(str))
    df["carpeta_roi"] = df["cropped image file path"].str.split("/").str[0]
    df["particion_oficial"] = particion
    return df

tr = cargar("mass_case_description_train_set.csv", "train")
te = cargar("mass_case_description_test_set.csv", "test")
df = pd.concat([tr, te], ignore_index=True)

print("Filas:", len(df), "| lesion_id único:", df.lesion_id.is_unique,
      "| filas duplicadas:", df.duplicated().sum())

## EDA tabular

In [ ]:
print(pd.crosstab(df.particion_oficial, df.pathology, margins=True))
print(pd.crosstab(df.particion_oficial, df.label, normalize="index").round(3))

In [ ]:
lpp = df.groupby("patient_id").size()
print("Pacientes:", df.patient_id.nunique())
print("Lesiones por paciente:", lpp.value_counts().sort_index().to_dict())
print("Imágenes completas distintas:", df["image file path"].nunique(), "de", len(df), "lesiones")

In [ ]:
print(pd.crosstab(df.particion_oficial, df["image view"]))
print(pd.crosstab(df.particion_oficial, df["left or right breast"]))
print(pd.crosstab(df.particion_oficial, df.breast_density))
print(df.groupby("breast_density").label.mean().round(3).to_dict())

In [ ]:
print(df.isna().sum()[df.isna().sum() > 0])
print(pd.crosstab(df.assessment, df.label))
print(pd.crosstab(df.subtlety, df.label))

In [ ]:
import os

# Carpeta de resultados dentro de tu Drive (usando tu variable DATA)
RESULTADOS = RES
os.makedirs(RESULTADOS, exist_ok=True)

# Lesiones por paciente
lpp = df.groupby("patient_id").size()

fig, ax = plt.subplots(1, 3, figsize=(15, 4))

pd.crosstab(df.particion_oficial, df.pathology).plot.bar(ax=ax[0], rot=0)
ax[0].set_title("Patología por partición oficial")

lpp.value_counts().sort_index().plot.bar(ax=ax[1], rot=0)
ax[1].set_title("Lesiones por paciente")
ax[1].set_xlabel("n.º de lesiones")

pd.crosstab(df.assessment, df.pathology).plot.bar(ax=ax[2], rot=0)
ax[2].set_title("assessment vs patología")

plt.tight_layout()
plt.savefig(f"{RESULTADOS}/eda_tabular.png", dpi=120, bbox_inches="tight")
plt.show()

# Comprobación: debe aparecer eda_tabular.png
print(os.listdir(RESULTADOS))

## Índice de los DICOM por encabezado (sin `force=True`)

In [ ]:
!pip install -q pydicom

import os, pydicom, pandas as pd
from tqdm.auto import tqdm

CACHE = os.path.join(ROOT, "indice_dicom.csv")

rutas = [os.path.join(d, f) for d, _, fs in os.walk(ROOT)
         for f in fs if f.lower().endswith(".dcm")]
print("Archivos .dcm encontrados:", len(rutas))

previo = (pd.read_csv(CACHE) if os.path.exists(CACHE)
          else pd.DataFrame(columns=["path", "series_uid", "sop_uid", "rows", "cols"]))
hechos = set(previo.path)

nuevos, fallidos = [], []
for p in tqdm([r for r in rutas if r not in hechos]):
    try:
        ds = pydicom.dcmread(p, stop_before_pixels=True)
        nuevos.append(dict(path=p, series_uid=str(ds.SeriesInstanceUID),
                           sop_uid=str(ds.SOPInstanceUID),
                           rows=int(ds.Rows), cols=int(ds.Columns)))
    except Exception as e:
        fallidos.append(p)
        print("No se pudo leer:", p, e)

indice = pd.concat([previo, pd.DataFrame(nuevos)], ignore_index=True)
indice.to_csv(CACHE, index=False)
print("Índice guardado:", len(indice), "archivos | lecturas fallidas nuevas:", len(fallidos))

## Cruce con el CSV: lesiones con serie ROI completa

In [ ]:
# Series de cada lesión, según el CSV (3.er componente de la ruta)
df["serie_crop"] = df["cropped image file path"].str.split("/").str[2]
df["serie_mask"] = df["ROI mask file path"].str.split("/").str[2]

# Archivos ilegibles: se excluyen de forma explícita
indice["area"] = indice.rows * indice.cols
ilegibles = indice[(indice.series_uid == "desconocido") | (indice.rows == 0)]
print("Archivos ilegibles excluidos:", len(ilegibles))

idx = (indice.drop(ilegibles.index)
             .drop_duplicates(["series_uid", "sop_uid"]).copy())
print("Archivos válidos:", len(idx), "| series:", idx.series_uid.nunique())
print("Archivos por serie:", idx.groupby("series_uid").size().value_counts().to_dict())

en_disco = set(idx.series_uid)
df["tiene_roi"] = df.serie_crop.isin(en_disco) | df.serie_mask.isin(en_disco)

por_serie = {s: g for s, g in idx.groupby("series_uid")}
filas = []
for r in df[df.tiene_roi].itertuples():
    partes = [por_serie[s] for s in {r.serie_crop, r.serie_mask} if s in por_serie]
    g = pd.concat(partes).sort_values("area")       # el menor área es el recorte
    k = len(g)
    ok = (k == 2) and (g.area.iloc[0] < 0.5 * g.area.iloc[1])
    filas.append(dict(lesion_id=r.lesion_id, n_roi_files=k, ok=ok,
                      crop_path=g.path.iloc[0],
                      mask_path=g.path.iloc[-1] if k >= 2 else None,
                      crop_dims=f"{g.rows.iloc[0]}x{g.cols.iloc[0]}"))

disp = pd.DataFrame(filas).merge(
    df[["lesion_id", "patient_id", "label", "particion_oficial"]], on="lesion_id")

print("\nLesiones con ROI:", len(disp), "| completas (recorte + máscara):", int(disp.ok.sum()))
print("Archivos ROI por lesión:", disp.n_roi_files.value_counts().to_dict())
print(disp[disp.ok].groupby("particion_oficial").agg(
    lesiones=("lesion_id", "size"), pacientes=("patient_id", "nunique"), malignas=("label", "sum")))
print("\nLesiones NO completas (primeras 5):")
print(disp[~disp.ok].head()[["lesion_id", "n_roi_files", "crop_dims"]])

In [ ]:
bien = disp[disp.ok]
bien.lesion_id.to_csv(f"{RES}/lesiones_descargadas.txt", index=False, header=False)
bien.to_csv(f"{RES}/lesiones_disponibles.csv", index=False)
pd.DataFrame({"path": list(ilegibles.path) + fallidos}).to_csv(
    f"{RES}/archivos_ilegibles.csv", index=False)
print(f"{len(bien)} lesiones guardadas en {RES}")

In [ ]:
print("Lesiones en la lista:", len(open(f"{RES}/lesiones_descargadas.txt").read().split()))

In [ ]:
# Diagnóstico de los archivos que no se pudieron leer (inconveniente real del día 2):
# eran archivos de 0 bytes sin la firma DICM en el byte 128.
malos = pd.DataFrame({"path": list(ilegibles.path) + fallidos})
malos["nombre"] = malos.path.apply(os.path.basename)
malos["bytes"] = malos.path.apply(os.path.getsize)
def firma(p):
    with open(p, "rb") as f:
        f.seek(128); return f.read(4)
malos["tiene_DICM"] = malos.path.apply(lambda p: firma(p) == b"DICM")
print(malos[["nombre", "bytes", "tiene_DICM"]].to_string())

def tiene_dicm(p):
    with open(p, "rb") as f:
        f.seek(128); return f.read(4) == b"DICM"
print("Archivos válidos sin cabecera DICM:", int((~idx.path.apply(tiene_dicm)).sum()), "de", len(idx))

## Partición por paciente (semilla 42)

In [ ]:
!python "{SRC}/01_particion.py" --data "{DATA}" --out "{RES}/particiones"     --modo propio --semilla 42 --disponibles "{RES}/lesiones_descargadas.txt"